In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("UGov_Data.csv", low_memory=False)

print("Dataset shape:", df.shape)
print("Minimum qweek:", df["week_num"].min())
print("Maximum qweek:", df["week_num"].max())

Dataset shape: (53833, 44)
Minimum qweek: 1
Maximum qweek: 54


In [2]:
# Keeping only qweeks 31 to 42
model_df = df[df["week_num"].between(31, 42)].copy()

# the period variable
model_df["surge_period"] = np.where(
    model_df["week_num"].between(31, 36),
    0,
    1
)

# Add readable labels
model_df["surge_period_label"] = model_df["surge_period"].map({
    0: "Earlier comparison period (qweeks 31–36)",
    1: "Later surge period (qweeks 37–42)"
})

print("Modelling dataset shape:", model_df.shape)

print(
    model_df.groupby("surge_period_label")["week_num"]
    .agg(["min", "max", "count"])
)

Modelling dataset shape: (12025, 46)
                                          min  max  count
surge_period_label                                       
Earlier comparison period (qweeks 31–36)   31   36   5995
Later surge period (qweeks 37–42)          37   42   6030


In [3]:
# Define the required columns
mask_outcome = "wore_mask_outside_home_adherent"
weight_col = "survey_weight"

# Check whether the weights contain any problems
print("Missing survey weights:", model_df[weight_col].isna().sum())
print("Zero or negative weights:", (model_df[weight_col] <= 0).sum())

# Keep the variables needed for this analysis
analysis_df = model_df[
    [
        "surge_period",
        "surge_period_label",
        mask_outcome,
        weight_col
    ]
].dropna().copy()

# Keep only valid positive survey weights
analysis_df = analysis_df[analysis_df[weight_col] > 0].copy()

# Multiply adherence by the survey weight
analysis_df["weighted_adherent"] = (
    analysis_df[mask_outcome] * analysis_df[weight_col]
)

# Create the period summary
period_summary = (
    analysis_df
    .groupby(
        ["surge_period", "surge_period_label"],
        observed=True
    )
    .agg(
        responses=(mask_outcome, "size"),
        unweighted_adherence=(mask_outcome, "mean"),
        total_survey_weight=(weight_col, "sum"),
        weighted_adherent_total=("weighted_adherent", "sum")
    )
    .reset_index()
    .sort_values("surge_period")
)

# Convert the results into percentages
period_summary["unweighted_adherence_percent"] = (
    period_summary["unweighted_adherence"] * 100
)

period_summary["weighted_adherence_percent"] = (
    period_summary["weighted_adherent_total"]
    / period_summary["total_survey_weight"]
    * 100
)

# Display the useful columns
display(
    period_summary[
        [
            "surge_period_label",
            "responses",
            "unweighted_adherence_percent",
            "weighted_adherence_percent"
        ]
    ].round(2)
)

# Calculate the weighted percentage-point change
earlier_rate = period_summary.loc[
    period_summary["surge_period"] == 0,
    "weighted_adherence_percent"
].iloc[0]

later_rate = period_summary.loc[
    period_summary["surge_period"] == 1,
    "weighted_adherence_percent"
].iloc[0]

percentage_point_change = later_rate - earlier_rate

print(
    "\nWeighted adherence change:",
    round(percentage_point_change, 2),
    "percentage points"
)

Missing survey weights: 0
Zero or negative weights: 0


,surge_period_label,responses,unweighted_adherence_percent,weighted_adherence_percent
0,Earlier comparison period (qweeks 31–36),5995,51.31,52.55
1,Later surge period (qweeks 37–42),6030,79.32,78.35



Weighted adherence change: 25.8 percentage points


## Model 1 : Adherence ~ surge_period


In [4]:
import statsmodels.api as sm
import numpy as np

# Outcome variable
y = analysis_df[mask_outcome]

# Predictor variable
X = analysis_df[["surge_period"]]

# Add the intercept
X = sm.add_constant(X)

# Fit a survey-weighted logistic regression
model_1 = sm.GLM(
    y,
    X,
    family=sm.families.Binomial(),
    freq_weights=analysis_df[weight_col]
).fit(cov_type="HC1")

# Display the full model output
print(model_1.summary())

                        Generalized Linear Model Regression Results                        
Dep. Variable:     wore_mask_outside_home_adherent   No. Observations:                12025
Model:                                         GLM   Df Residuals:                 12023.01
Model Family:                             Binomial   Df Model:                            1
Link Function:                               Logit   Scale:                          1.0000
Method:                                       IRLS   Log-Likelihood:                -7298.2
Date:                             Mon, 27 Jul 2026   Deviance:                       14596.
Time:                                     19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                  4   Pseudo R-squ. (CS):            0.07220
Covariance Type:                               HC1                                         
                   coef    std err          z      P>|z|      [0.025      0.975]

In [5]:
# Extract the surge-period result
coefficient = model_1.params["surge_period"]
p_value = model_1.pvalues["surge_period"]

# Convert the coefficient into an odds ratio
odds_ratio = np.exp(coefficient)

# Calculate the 95% confidence interval for the odds ratio
confidence_interval = model_1.conf_int().loc["surge_period"]
ci_lower = np.exp(confidence_interval[0])
ci_upper = np.exp(confidence_interval[1])

print("Coefficient:", round(coefficient, 4))
print("Odds ratio:", round(odds_ratio, 2))
print(
    "95% confidence interval:",
    round(ci_lower, 2),
    "to",
    round(ci_upper, 2)
)
print("P-value:", p_value)

Coefficient: 1.1839
Odds ratio: 3.27
95% confidence interval: 3.02 to 3.54
P-value: 3.823008718148163e-187


In [6]:
# Create the dataset needed for the state model
state_analysis_df = model_df[
    [
        "state",
        "surge_period",
        "surge_period_label",
        weight_col
    ]
].dropna().copy()

# Keep only positive survey weights
state_analysis_df = state_analysis_df[
    state_analysis_df[weight_col] > 0
].copy()

print("Rows available for state analysis:", len(state_analysis_df))
print("Missing state values:", model_df["state"].isna().sum())

print("\nState categories:")
print(sorted(state_analysis_df["state"].unique()))

print("\nResponse counts by state and period:")

state_counts = pd.crosstab(
    state_analysis_df["state"],
    state_analysis_df["surge_period_label"],
    margins=True
)

display(state_counts)

Rows available for state analysis: 12025
Missing state values: 0

State categories:
['Australian Capital Territory', 'New South Wales', 'Northern Territory', 'Queensland', 'South Australia', 'Tasmania', 'Victoria', 'Western Australia']

Response counts by state and period:


surge_period_label,Earlier comparison period (qweeks 31–36),Later surge period (qweeks 37–42),All
state,,,
Australian Capital Territory,87,99,186
New South Wales,1802,1864,3666
Northern Territory,46,41,87
Queensland,1197,1187,2384
South Australia,614,596,1210
Tasmania,121,123,244
Victoria,1535,1555,3090
Western Australia,593,565,1158
All,5995,6030,12025


## Model 2 : Adherence ~ state + surge_period

In [7]:
import statsmodels.formula.api as smf

model_2 = smf.glm(
    formula="""
        wore_mask_outside_home_adherent
        ~ surge_period
        + C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_2.summary())

                        Generalized Linear Model Regression Results                        
Dep. Variable:     wore_mask_outside_home_adherent   No. Observations:                12025
Model:                                         GLM   Df Residuals:                 12016.01
Model Family:                             Binomial   Df Model:                            8
Link Function:                               Logit   Scale:                          1.0000
Method:                                       IRLS   Log-Likelihood:                -6271.5
Date:                             Mon, 27 Jul 2026   Deviance:                       12543.
Time:                                     19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                  5   Pseudo R-squ. (CS):             0.2178
Covariance Type:                               HC1                                         
                                                                                

## Model 3: Interaction between surge period and state

In [8]:
model_3 = smf.glm(
    formula="""
        wore_mask_outside_home_adherent
        ~ surge_period
        * C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_3.summary())

                        Generalized Linear Model Regression Results                        
Dep. Variable:     wore_mask_outside_home_adherent   No. Observations:                12025
Model:                                         GLM   Df Residuals:                 12009.01
Model Family:                             Binomial   Df Model:                           15
Link Function:                               Logit   Scale:                          1.0000
Method:                                       IRLS   Log-Likelihood:                -5940.3
Date:                             Mon, 27 Jul 2026   Deviance:                       11881.
Time:                                     19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                  5   Pseudo R-squ. (CS):             0.2598
Covariance Type:                               HC1                                         
                                                                                

In [9]:
## Avoiding Guests at Home

outcome = "avoided_having_guests_at_home_adherent"

model_guests = smf.glm(
    formula=f"""
        {outcome}
        ~ surge_period
        * C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_guests.summary())

                           Generalized Linear Model Regression Results                            
Dep. Variable:     avoided_having_guests_at_home_adherent   No. Observations:                12025
Model:                                                GLM   Df Residuals:                 12009.01
Model Family:                                    Binomial   Df Model:                           15
Link Function:                                      Logit   Scale:                          1.0000
Method:                                              IRLS   Log-Likelihood:                -7015.2
Date:                                    Mon, 27 Jul 2026   Deviance:                       14030.
Time:                                            19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                         5   Pseudo R-squ. (CS):             0.1311
Covariance Type:                                      HC1                                         
          

In [10]:
## Avoiding Small Social Gatherings

outcome = "avoided_small_social_gatherings_adherent"

model_small = smf.glm(
    formula=f"""
        {outcome}
        ~ surge_period
        * C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_small.summary())

                            Generalized Linear Model Regression Results                             
Dep. Variable:     avoided_small_social_gatherings_adherent   No. Observations:                12025
Model:                                                  GLM   Df Residuals:                 12009.01
Model Family:                                      Binomial   Df Model:                           15
Link Function:                                        Logit   Scale:                          1.0000
Method:                                                IRLS   Log-Likelihood:                -7320.6
Date:                                      Mon, 27 Jul 2026   Deviance:                       14641.
Time:                                              19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                           4   Pseudo R-squ. (CS):             0.1324
Covariance Type:                                        HC1                                

In [11]:
## Avoiding Medium Social Gatherings

outcome = "avoided_medium_social_gatherings_adherent"

model_medium = smf.glm(
    formula=f"""
        {outcome}
        ~ surge_period
        * C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_medium.summary())

                             Generalized Linear Model Regression Results                             
Dep. Variable:     avoided_medium_social_gatherings_adherent   No. Observations:                12025
Model:                                                   GLM   Df Residuals:                 12009.01
Model Family:                                       Binomial   Df Model:                           15
Link Function:                                         Logit   Scale:                          1.0000
Method:                                                 IRLS   Log-Likelihood:                -7075.7
Date:                                       Mon, 27 Jul 2026   Deviance:                       14151.
Time:                                               19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                            5   Pseudo R-squ. (CS):             0.1283
Covariance Type:                                         HC1                      

In [12]:

## Avoiding Large Social Gatherings

outcome = "avoided_large_social_gatherings_adherent"

model_large = smf.glm(
    formula=f"""
        {outcome}
        ~ surge_period
        * C(state, Treatment(reference='New South Wales'))
    """,
    data=model_df,
    family=sm.families.Binomial(),
    freq_weights=model_df["survey_weight"]
).fit(cov_type="HC1")

print(model_large.summary())


                            Generalized Linear Model Regression Results                             
Dep. Variable:     avoided_large_social_gatherings_adherent   No. Observations:                12025
Model:                                                  GLM   Df Residuals:                 12009.01
Model Family:                                      Binomial   Df Model:                           15
Link Function:                                        Logit   Scale:                          1.0000
Method:                                                IRLS   Log-Likelihood:                -6628.9
Date:                                      Mon, 27 Jul 2026   Deviance:                       13258.
Time:                                              19:37:58   Pearson chi2:                 1.20e+04
No. Iterations:                                           5   Pseudo R-squ. (CS):             0.1093
Covariance Type:                                        HC1                                

In [13]:
from scipy.stats import chi2

outcomes = {
    "Mask wearing":
        "wore_mask_outside_home_adherent",

    "Avoiding guests":
        "avoided_having_guests_at_home_adherent",

    "Avoiding small gatherings":
        "avoided_small_social_gatherings_adherent",

    "Avoiding medium gatherings":
        "avoided_medium_social_gatherings_adherent",

    "Avoiding large gatherings":
        "avoided_large_social_gatherings_adherent"
}

final_models = {
    "Mask wearing": model_3,
    "Avoiding guests": model_guests,
    "Avoiding small gatherings": model_small,
    "Avoiding medium gatherings": model_medium,
    "Avoiding large gatherings": model_large
}

interaction_results = []

for behaviour, outcome_column in outcomes.items():

    reduced_model = smf.glm(
        formula=f"""
            {outcome_column}
            ~ surge_period
            + C(state, Treatment(reference='New South Wales'))
        """,
        data=model_df,
        family=sm.families.Binomial(),
        freq_weights=model_df["survey_weight"]
    ).fit(cov_type="HC1")

    final_model = final_models[behaviour]

    lr_statistic = 2 * (
        final_model.llf - reduced_model.llf
    )

    df_difference = int(
        final_model.df_model - reduced_model.df_model
    )

    p_value = chi2.sf(
        lr_statistic,
        df_difference
    )

    interaction_results.append({
        "Behaviour": behaviour,
        "LR statistic": round(lr_statistic, 2),
        "Degrees of freedom": df_difference,
        "P-value": (
            "<0.001"
            if p_value < 0.001
            else round(p_value, 3)
        )
    })

interaction_test_table = pd.DataFrame(interaction_results)

display(interaction_test_table)

,Behaviour,LR statistic,Degrees of freedom,P-value
0,Mask wearing,662.42,7,<0.001
1,Avoiding guests,338.32,7,<0.001
2,Avoiding small gatherings,286.52,7,<0.001
3,Avoiding medium gatherings,338.39,7,<0.001
4,Avoiding large gatherings,345.28,7,<0.001


In [14]:
# Save complete Model 3 coefficient tables and model-comparison results

from pathlib import Path

output_dir = Path("../graphs and tables")
output_dir.mkdir(parents=True, exist_ok=True)

model_output_files = {
    "Mask wearing": "model3_mask_wearing.csv",
    "Avoiding guests": "model3_avoiding_guests.csv",
    "Avoiding small gatherings": "model3_avoiding_small_gatherings.csv",
    "Avoiding medium gatherings": "model3_avoiding_medium_gatherings.csv",
    "Avoiding large gatherings": "model3_avoiding_large_gatherings.csv"
}

for behaviour, model in final_models.items():
    confidence_intervals = model.conf_int()

    coefficient_table = pd.DataFrame({
        "term": model.params.index,
        "coefficient": model.params.values,
        "standard_error": model.bse.values,
        "odds_ratio": np.exp(model.params.values),
        "odds_ratio_ci_lower": np.exp(confidence_intervals[0].values),
        "odds_ratio_ci_upper": np.exp(confidence_intervals[1].values),
        "p_value": model.pvalues.values
    })

    coefficient_table.to_csv(
        output_dir / model_output_files[behaviour],
        index=False
    )

interaction_test_table.to_csv(
    output_dir / "model_comparison_likelihood_ratio_tests.csv",
    index=False
)

print("Model coefficient and comparison tables saved.")

Model coefficient and comparison tables saved.
